### Import Library

In [6]:
# Import library 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns
#
# from scipy import stats

import warnings

### Set option in pandas to display all rows and columns in dataset

In [7]:
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows", None)

### Ignore library Deprecated warning messages

In [8]:
# def ignore_warning(): # define function 
#     return warnings.filterwarnings(action='ignore', category= DeprecationWarning, module=r'.*randpool')

# ignore_warning() # function call
warnings.filterwarnings('ignore')


### Load Dataset

In [9]:
def load_dataset(dataset_url): # define function
    return  pd.read_csv(dataset_url) # to read csv data
 
# call function load_dataset() and store dataset value in population_dataframe
population_dataframe = load_dataset(r'./estimated_population.csv')

FileNotFoundError: [Errno 2] No such file or directory: './estimated_population.csv'

# EDA (Exploratory Data Analysis)

In [ ]:
# Dimensions of dataset
population_dataframe.shape

In [ ]:
# Gives top 5 rows of dataset
population_dataframe.tail(10)

In [ ]:
# Gives bottom 5 rows of dataset
population_dataframe.tail()

In [ ]:
# Gives random sample data
population_dataframe.sample(10)

In [ ]:
# rename columns name i.e no space should be in columns name
# use of try and except to handle if value list has not appropriate length for column
new_column_name = ['statistic_label','year','age_group','sex','region','unit','value']

def columns_dict(cols):
    new_col = {}
    
    if len(cols) == len(population_dataframe.columns):
        try:
            for i in range(len(cols)):
                new_col[population_dataframe.columns[i]] = cols[i]
        except:
            print("An error occurred while creating the column dictionary.")
    else:
        print(f"Your length of Population Dataframe {len(population_dataframe.columns)} != {len(new_column_name)} with length of New Column ")
    
    return new_col

col_dict = columns_dict(new_column_name)


### Rename Column names

In [ ]:

# inplace = True set the values permenantly so that we can use new columns name
population_dataframe.rename(columns=col_dict, inplace=True)

In [ ]:
# drop column STATISTIC Label and Unit because it
population_dataframe.drop({'statistic_label', 'unit'}, axis=1, inplace=True)
population_dataframe.head()

In [ ]:
# Checking the types of data
population_dataframe.dtypes

In [ ]:
# isna().sum() provide sum of null value in each column in datasets
population_dataframe.isnull().sum()

In [ ]:
# info() provide information about datasets
population_dataframe.info()

In [ ]:
population_dataframe.count()

In [ ]:
# count duplicated value in dataset
population_dataframe.duplicated().sum()

In [ ]:
# describe() generate statistical descriptive value of dataset
population_dataframe.describe()

In [ ]:
# describe(include=object) generate statistical descriptive value for object datatype of dataset
population_dataframe.describe(include=object)

In [ ]:
# It return all unique value of series object
population_dataframe['age_group'].unique()

In [ ]:
# It return all unique value of series object
population_dataframe['year'].unique()

#### Remove State from observation

In [ ]:
# Remove State observation from region because state is sum of all other 8 region population

new_population_dataframe = population_dataframe.loc[population_dataframe['region'] != 'State']
new_population_dataframe.head()

In [ ]:
# Unique region 
new_population_dataframe['region'].unique()

In [ ]:
# Unique sex 
new_population_dataframe['sex'].unique()

In [ ]:
# Unique age_group
new_population_dataframe['age_group'].unique()

#### Remove Both sexes and All age group from observation

In [ ]:
# Remove Both sexes observation from sex column and All ages observation from age_group column because both sexes is sum of male and female population and all ages are sum of rest of categorical age

new_population_dataframe = new_population_dataframe.loc[(new_population_dataframe['sex'] != 'Both sexes') & (new_population_dataframe['age_group'] != 'All ages')]
new_population_dataframe.reset_index(drop=True, inplace=True)
new_population_dataframe.tail(5)

In [ ]:
# Boxplot helps to identify outliers
def plot_boxplot(col_name):
    plt.figure(figsize=(8,6))
    sns.boxplot(x=col_name);

In [ ]:
# call plot_boxplot for year column, Hence there are no outliers
plot_boxplot(new_population_dataframe['year'])

In [ ]:
# call plot_boxplot function for value column, Hence there are outliers
plot_boxplot(new_population_dataframe['value'])

In [ ]:
# Function to remove outliers from datasets by using IQR (Interquartile Range)

def remove_outliers(col_name):
    Q1 = new_population_dataframe[col_name].quantile(0.25)
    Q3 = new_population_dataframe[col_name].quantile(0.75)
    IQR = Q3-Q1 
    lower_limit = Q1 - 1.5*IQR
    upper_limit = Q3 + 1.5*IQR
    print('IQR',IQR)
    print('lower_limit',lower_limit)
    print('upper_limit',upper_limit)
    new_data = new_population_dataframe[(new_population_dataframe['value'] > lower_limit) & (new_population_dataframe['value'] < upper_limit)]
    return new_data
    
newdata_no_outliers = remove_outliers('value')

plot_boxplot(newdata_no_outliers['value'])


In [ ]:
plt.figure(figsize=(8,6))
# correlation = population_dataframe.corr()
correlation = newdata_no_outliers.corr(numeric_only=True)
sns.heatmap(correlation, annot=True, fmt='.2f', linewidths=2, cmap="BrBG")
correlation


In [ ]:
newdata_no_outliers.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(newdata_no_outliers["year"], newdata_no_outliers["value"])
# ax.scatter(population_dataframe["year"], population_dataframe["value"])

ax.set_xlabel("Years")
ax.set_ylabel("Value in thousands")
plt.show()

In [ ]:
newdata_no_outliers.describe()

In [ ]:
# Histplot explain distribution of data i.e Symmetric distribution  data
sns.histplot(data=newdata_no_outliers, x='value', kde=True, bins=50);

#### Calculate Skewness for distribution of data

In [ ]:
# best skew value should be -3 to +3
skew_value = stats.skew(newdata_no_outliers['value'], axis=0, bias=True)
print('Skew Value',skew_value)

#### Calculate kurtosis for distribution of data

In [ ]:
# best kurtosis value should be -10 to +10
kurtosis_value = stats.kurtosis(newdata_no_outliers['value'], axis=0, bias=True)
print('kurtosis Value',kurtosis_value)

In [ ]:
newdata_no_outliers.describe(include=object)

In [ ]:
newdata_no_outliers['age_group'].unique()

# Statistical Data Analysis and visualization

### Total population by year of all states from 2011-2023

In [ ]:

all_age_bt_sex_state_df = population_dataframe.loc[(population_dataframe['sex'] == 'Both sexes') & (population_dataframe['region'] == 'State') & (population_dataframe['age_group'] == 'All ages')].reset_index(drop=True)


In [ ]:
# Statistical value for bar graph year and population
all_age_bt_sex_state_df[['year','value']]

#### Statistical value  for bar graph

In [ ]:
print(f"Max: {all_age_bt_sex_state_df[['year','value']].max()},  Min: {all_age_bt_sex_state_df[['year','value']].min()}, Mean: {all_age_bt_sex_state_df[['year','value']].mean()} ")


In [ ]:
#Bar graph to compare population by year 2011-2023 
plt.figure(figsize=(8,6))
plt.bar(all_age_bt_sex_state_df['year'], all_age_bt_sex_state_df['value'])
plt.title("Total population of Ireland")
plt.ylabel("Population in thousands")
plt.xlabel("Years")
plt.show()

### Child population age 0-4 years, both sex, all states

In [ ]:

age04_bt_sex_state_df = population_dataframe.loc[(population_dataframe['sex'] == 'Both sexes') & (population_dataframe['region'] == 'State') & (population_dataframe['age_group'] == '0 - 4 years')].reset_index(drop=True)
age04_bt_sex_state_df

In [ ]:
#Bar graph to compare population by year 2011-2023 
plt.figure(figsize=(8,6))
plt.plot(age04_bt_sex_state_df['year'], age04_bt_sex_state_df['value'], marker='*')
plt.title("Population of child from age 0-4 years")
plt.ylabel("Population in thousands")
plt.xlabel("Years")
plt.grid(color = 'grey', linestyle = '--', linewidth = 0.3)
plt.show()

# why child birth is decreasing each year -> research

###  Older Adults Population age 85 years and above , both sex, all states

In [ ]:
age85_above_bt_sex_state_df = population_dataframe.loc[(population_dataframe['sex'] == 'Both sexes') & (population_dataframe['region'] == 'State') & (population_dataframe['age_group'] == '85 years and over')].reset_index(drop=True)
age85_above_bt_sex_state_df

In [ ]:
#Bar graph to compare population by year 2011-2023 
plt.figure(figsize=(8,6))
plt.plot(age85_above_bt_sex_state_df['year'], age85_above_bt_sex_state_df['value'], marker='1')
plt.title("Population age 80 and above")
plt.ylabel("Population in thousands")
plt.xlabel("Years")
plt.grid(color = 'grey', linestyle = '--', linewidth = 0.3)
plt.show()

#### Define Class with method get_age_data() to get data frame based on  different age group

In [ ]:
# Define class name PopulationByAgeGroup
class PopulationByAgeGroup:
    
    # __init__() is Constructor: The function that runs whenever we make a new object i.e population_data 
    def __init__(self, population_dataframe):
        self.population_dataframe = population_dataframe
        
        #gives uniques all age group
        self.age_group_only = population_dataframe['age_group'].unique()
        print('All age group: ',self.age_group_only)
        
    def get_age_data(self, start_age, end_age, sex, region):
        age_group_years = self.age_group_only[start_age: end_age]
        print('age_group_years',age_group_years)

        # find either value from different age exist or not 
        has_age_group = population_dataframe['age_group'].isin(age_group_years)
        
        # filter data age lies from '15 - 19 years' to  '60 - 64 years'
        filter_age_group_df = (population_dataframe.loc[has_age_group])
        age_group_df = filter_age_group_df.loc[(filter_age_group_df['sex'] == sex) & (filter_age_group_df['region'] == region)]
        age_group_df = age_group_df.groupby(['year']).sum().reset_index()
        return age_group_df


# Initilization of Class PopulationByAgeGroup
population_data = PopulationByAgeGroup(population_dataframe)

### Working population in ireland age 15-64

In [ ]:
# get dataframe of age group from '15 - 19 years' to  '60 - 64 years'
# Call PopulationByAgeGroup
working_age_data = population_data.get_age_data(start_age = 3, end_age= 13, sex='Both sexes', region='State')
print('Woring population in Ireland by year : \n',working_age_data)


In [ ]:
# function to plot line chart to show trend of population based on age group
def plot_linegraph(data_frame, title,  marker_icon):
    plt.figure(figsize=(8,6))
    plt.plot(data_frame['year'], data_frame['value'], marker=marker_icon)
    plt.title(title)
    plt.ylabel("Population in thousands")
    plt.xlabel("Years")
    plt.grid(color = 'grey', linestyle = '--', linewidth = 0.3)
    plt.show()
    
# line plot showing trend of working populatin in ireland by years
plot_linegraph(working_age_data, "Population age group between 15 to 64 which is working age group of Irland", "o")

### Children population in ireland age range  '0 - 4 years' to '15 - 19 years'

In [ ]:
# get dataframe of age group from '0-4 years' to  '10 - 14 years'
# Call PopulationByAgeGroup
children_age_data = population_data.get_age_data(start_age = 0, end_age= 3, sex='Both sexes', region='State')
print('Children population in Ireland by year : \n',children_age_data)


In [ ]:
# line plot showing trend of working populatin in ireland by years
plot_linegraph(children_age_data, "Population age group between 0 to 14 which is children age group of Irland", "1")

### Older age population in ireland age range 65 and above

In [ ]:
# Call PopulationByAgeGroup
older_age_data = population_data.get_age_data(start_age = 13, end_age= -1, sex='Both sexes', region='State')
print('Older population in Ireland by year : \n',working_age_data)

### Chart data for Children, Working age and Older age for year 2023

In [10]:
chart_data_frame = [children_age_data, working_age_data, older_age_data]
chart_data_frame = pd.concat([children_age_data['year'], children_age_data['value'], working_age_data['value'], older_age_data['value']], keys=['year','children','working','older'], axis=1).reset_index(drop=True)
chart_data_frame

NameError: name 'children_age_data' is not defined

In [ ]:
chart_data_frame = chart_data_frame[chart_data_frame['year'] == 2023].reset_index(drop=True)
chart_data_frame

In [12]:
pie_data = chart_data_frame.iloc[:,1:].values.flatten()
years = chart_data_frame.iloc[:, 0].values
plt.figure(figsize=(8,6))
plt.pie(pie_data, labels=['Children', 'Working People', 'Older'], shadow=False, startangle=90, autopct='%1.1f%%', wedgeprops={'edgecolor': '#ffffff'} )
plt.title(f'R atio of Children,  Working people and Older age people in year {years}')
plt.tight_layout()
plt.show()

NameError: name 'chart_data_frame' is not defined

In [13]:
# Filter data for 'Male' and 'Female' sexes
filter_mf_df = newdata_no_outliers[(newdata_no_outliers['sex'] == 'Male') | (newdata_no_outliers['sex'] == 'Female')]
male_female_df = filter_mf_df.groupby(['year','region','sex'])['value'].sum()
male_female_df


NameError: name 'newdata_no_outliers' is not defined